# Macro Portfolio Local v1.0.13 — System Runner

This notebook is intentionally thin. **Production logic lives only in `src/macro_portfolio/`.**
Running all cells creates a fresh runtime, refreshes current data, and recomputes B→K; Section L runs as a non-blocking diagnostic.

Do not run downstream sections manually. If a production section fails, fix the cause and restart the kernel / Run All.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(
        f'MacroPortfolioLocal requires Python 3.11 or 3.12; current kernel is {sys.version.split()[0]}. '
        'Change Jupyter to your Python 3.11 kernel before running.'
    )

# Lightweight dependency preflight. Production/model logic is never duplicated here.
_REQUIRED_IMPORTS = [
    'numpy', 'pandas', 'scipy', 'pyarrow', 'duckdb', 'yaml',
    'pandas_datareader', 'yfinance', 'fredapi', 'cvxpy', 'statsmodels',
    'sklearn', 'plotly', 'arch', 'hmmlearn', 'requests', 'bs4',
    'pandas_market_calendars', 'filelock', 'dotenv',
]
_missing = [name for name in _REQUIRED_IMPORTS if importlib.util.find_spec(name) is None]

# Find the extracted project root and expose bundled src/ without requiring pip install -e .
_start = Path.cwd().resolve()
PROJECT_ROOT = None
for _candidate in (_start, *_start.parents):
    if ((_candidate / 'pyproject.toml').exists() and (_candidate / 'src' / 'macro_portfolio').exists()):
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open RUN_ME.ipynb from the extracted MacroPortfolioLocal_v1_0_13_VALUATION_GUARDRAILS folder.')

if _missing:
    print('Installing missing runtime dependencies:', ', '.join(_missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')])
    importlib.invalidate_caches()

_src = PROJECT_ROOT / 'src'
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from macro_portfolio.notebook_runner import validate_notebook_environment, run_everything
PROJECT_ROOT = validate_notebook_environment()
print(f'✓ System runner ready: {PROJECT_ROOT}')
print('✓ FRED API credential loaded automatically from the private local package.')
print('✓ Production code source: src/macro_portfolio (single source of truth).')


In [ ]:
# FULL DYNAMIC RUN — always starts from a fresh runtime and refreshes upstream data.
RESULT = run_everything(include_diagnostic_l=True)
NS = RESULT['namespace']
print('\n' + '=' * 88)
print('FULL SYSTEM RUN COMPLETE')
print('=' * 88)
print('Run ID:', RESULT['run_id'])
print('Snapshot:', RESULT['run_dir'])
print('Section L:', NS.get('SECTION_L_STATUS', 'UNKNOWN'))


In [ ]:
# Compact live outputs after a successful production run.
from IPython.display import display
for _name in [
    'PM_MACRO_STATE_V2',
    'FINAL_SECTOR_SIGNAL_DASHBOARD',
    'F2_CURRENCY_RECONCILIATION_AUDIT',
    'F_FINAL_VALUATION_HANDOFF',
    'G_FINAL_PORTFOLIO_WEIGHTS',
    'K2_READINESS',
]:
    _obj = NS.get(_name)
    if _obj is not None:
        print(f'\n### {_name}')
        display(_obj)

# Model-risk exceptions are surfaced explicitly rather than hidden inside the full table.
_f = NS.get('F_FINAL_VALUATION_HANDOFF')
if _f is not None and 'valuation_critical_block' in _f.columns:
    _critical = _f[_f['valuation_critical_block'].fillna(False)].copy()
    print('\n### CRITICAL_VALUATION_REVIEW_QUEUE')
    if _critical.empty:
        print('None.')
    else:
        _cols = [c for c in [
            'ticker', 'company_name', 'current_price', 'expected_intrinsic_value',
            'iv_price_ratio', 'consensus_target_mean', 'consensus_divergence_abs',
            'price_currency', 'valuation_financial_currency', 'financial_to_price_fx',
            'valuation_share_units', 'valuation_anomaly_reason',
        ] if c in _critical.columns]
        display(_critical[_cols])
